# Milestone 4 — MCQ Task & Fine-Tuning

In [ ]:
# ======================================================
# PHASE 5 - STEP 16 : PROMPT GROUPS
# ======================================================

from sklearn.model_selection import GroupKFold

train_df["group"] = train_df["prompt"]

gkf = GroupKFold(n_splits=5)

splits = list(
    gkf.split(
        train_df,
        groups=train_df["group"]
    )
)

print(f"Total folds: {len(splits)}")

train_idx, valid_idx = splits[0]

print("Train size :", len(train_idx))
print("Valid size :", len(valid_idx))

In [ ]:
# ======================================================
# PHASE 5 - STEP 17 : CREATE FOLD DATASETS
# ======================================================

train_fold = tokenized_train.select(train_idx)
valid_fold = tokenized_train.select(valid_idx)

print(train_fold)
print(valid_fold)

In [ ]:
# Remove raw text columns
cols_to_remove = ["prompt", "A", "B", "C", "D", "E"]

train_fold = train_fold.remove_columns(cols_to_remove)
valid_fold = valid_fold.remove_columns(cols_to_remove)

print(train_fold)
print(valid_fold)

In [ ]:
# ======================================================
# PHASE 5 - STEP 18 : DATA COLLATOR
# ======================================================

from transformers import DataCollatorForMultipleChoice

data_collator = DataCollatorForMultipleChoice(
    tokenizer=tokenizer
)

print("Data Collator Ready")

In [ ]:
# ======================================================
# PHASE 5 - STEP 19 : LOAD MODEL
# ======================================================

from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

print("Model Loaded Successfully!")

In [ ]:
# ======================================================
# PHASE 5 - STEP 20 : TRAINING ARGUMENTS
# ======================================================

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_fold1",

    do_train=True,
    do_eval=True,

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    fp16=False,
    bf16=False,

    logging_steps=50,

    report_to="wandb",

    save_total_limit=1,

    remove_unused_columns=False,

    seed=42
)

print(training_args)

In [ ]:
# ======================================================
# PHASE 5 - STEP 21 : MAP@3 METRIC
# ======================================================

import numpy as np

def apk(actual, predicted, k=3):
    """
    Average Precision at K for a single sample
    """
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def map3(predictions, labels):
    """
    Mean Average Precision @ 3
    """
    scores = []

    for pred, label in zip(predictions, labels):
        scores.append(apk(label, pred, k=3))

    return np.mean(scores)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Top-3 predicted classes
    top3 = np.argsort(logits, axis=1)[:, ::-1][:, :3]

    score = map3(top3, labels)

    return {
        "map@3": score
    }

print("MAP@3 Metric Ready!")

In [ ]:
# ======================================================
# PHASE 5 - STEP 22 : CREATE TRAINER
# ======================================================

from transformers import Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_fold,
    eval_dataset=valid_fold,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer Ready!")

In [ ]:
# ======================================================
# PHASE 5 - STEP 23 : TRAIN MODEL
# ======================================================

train_result = trainer.train()

print("\nTraining Completed!")

In [ ]:
print(train_fold.features)

In [ ]:
sample = train_fold[0]

print(sample.keys())
print(sample["label"])

In [ ]:
batch = data_collator([train_fold[0], train_fold[1]])

for k, v in batch.items():
    print(k, v.shape, v.dtype)